# LangChain Multi-Agent System with Subagents pattern

In the subagents architecture, a central main agent (often referred to as a supervisor) coordinates subagents by calling them as tools. The main agent decides which subagent to invoke, what input to provide, and how to combine results. Subagents are stateless—they don’t remember past interactions, with all conversation memory maintained by the main agent. This provides context isolation: each subagent invocation works in a clean context window, preventing context bloat in the main conversation.

## Prerequisites

Deploy the infrastructure using the Terraform files in this directory (see `lang_chain.ipynb` for details):
```bash
terraform init
terraform apply -auto-approve
```

## 1. Install Dependencies

In [2]:
%pip install langchain langgraph langchain-openai langchain-mcp-adapters langgraph-supervisor

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
%pip install --upgrade pip
%pip install langchain-azure-ai

  Using cached pip-26.1.1-py3-none-any.whl.metadata (4.6 kB)
Using cached pip-26.1.1-py3-none-any.whl (1.8 MB)
  Attempting uninstall: pip
    Found existing installation: pip 25.3
    Uninstalling pip-25.3:
      Successfully uninstalled pip-25.3
Note: you may need to restart the kernel to use updated packages.
  Using cached langchain_azure_ai-1.2.3-py3-none-any.whl.metadata (18 kB)
  Using cached azure_ai_projects-2.1.0-py3-none-any.whl.metadata (61 kB)
Using cached azure_ai_projects-2.1.0-py3-none-any.whl (274 kB)

   ---------------------------------------- 0/4 [azure-ai-contentunderstanding]
   ---------------------------------------- 0/4 [azure-ai-contentunderstanding]
   ---------- ----------------------------- 1/4 [azure-ai-contentsafety]
   ---------- ----------------------------- 1/4 [azure-ai-contentsafety]
  Attempting uninstall: azure-ai-projects
   ---------- ----------------------------- 1/4 [azure-ai-contentsafety]
    Found existing installation: azure-ai-projects 2.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
azure-ai-agentserver-agentframework 1.0.0b13 requires agent-framework-core<=1.0.0b260107,>=1.0.0b251112, but you have agent-framework-core 1.0.1 which is incompatible.


## 2. Get the LLM and MCP Server Endpoints

Retrieve the FQDNs of the Gemma 4 model and the MCP web search server deployed on ACA from Terraform output.

In [4]:
aca_gemma4_31b_it_a100_fqdn = ! terraform -chdir=infra output -raw aca_gemma4_31b_it_a100_fqdn
aca_gemma4_31b_it_a100_fqdn = aca_gemma4_31b_it_a100_fqdn.n
print("LLM Endpoint:", aca_gemma4_31b_it_a100_fqdn)

foundry_endpoint = ! terraform -chdir=infra output -raw foundry_endpoint
foundry_endpoint = foundry_endpoint.n
print("Foundry Endpoint:", foundry_endpoint)

foundry_api_key = ! terraform -chdir=infra output -raw foundry_api_key
foundry_api_key = foundry_api_key.n
print("Foundry API Key:", f"{foundry_api_key[-10:]}...")  # Print only the last 10 characters for security

llm_model_deployment_name_chatgpt = ! terraform -chdir=infra output -raw llm_model_deployment_name_chatgpt
llm_model_deployment_name_chatgpt = llm_model_deployment_name_chatgpt.n
print("LLM Model Deployment Name (ChatGPT):", llm_model_deployment_name_chatgpt)

aca_mcp_server_open_web_search_fqdn = ! terraform -chdir=infra output -raw aca_mcp_server_open_web_search_fqdn
aca_mcp_server_open_web_search_fqdn = aca_mcp_server_open_web_search_fqdn.n
print("MCP Server Endpoint:", aca_mcp_server_open_web_search_fqdn)

LLM Endpoint: gemma-4-31b-it-a100.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io
Foundry Endpoint: https://foundry-555.cognitiveservices.azure.com/
Foundry API Key: AAACOGxtMj...
LLM Model Deployment Name (ChatGPT): gpt-5.4
MCP Server Endpoint: aca-mcp-server-open-web-search.gentlemushroom-793350b5.swedencentral.azurecontainerapps.io


## 3. Create the LLM Model

Create a `ChatOpenAI` model pointing at the vLLM OpenAI-compatible endpoint.

In [5]:
from langchain_openai import ChatOpenAI

# model = ChatOpenAI(
#     base_url=f"http://{aca_gemma4_31b_it_a100_fqdn}/v1",
#     api_key="EMPTY",
#     model="google/gemma-4-31B-it",
#     streaming=True,
#     max_completion_tokens=512
# )

model = ChatOpenAI(
    base_url=f"{foundry_endpoint}/openai/v1",
    api_key=foundry_api_key,
    model=llm_model_deployment_name_chatgpt,
    streaming=True,
    max_completion_tokens=512
)

Test the model with a simple prompt to ensure it’s working correctly.

In [6]:
from langchain_core.messages import HumanMessage

response = model.stream([HumanMessage(content="Tell me briefly about yourself.")])

for chunk in response:
    print(chunk.content, end="", flush=True)

I’m ChatGPT, an AI assistant created by OpenAI. I can help with writing, coding, explanations, brainstorming, summarizing, and answering questions across many topics.

I don’t have personal experiences or feelings, but I can hold a conversation and adapt to what you need—whether that’s quick answers or deeper help. If you want, I can also tell you what I’m especially good at or where my limits are.

## Deep Research Agent

The [Deep Research](https://docs.langchain.com/oss/python/deepagents/deep-research) pattern uses the `deepagents` package to build a multi-step web research agent that:

1. **Plans** research by decomposing the question into focused tasks (using a TODO list)
2. **Delegates** focused research tasks to sub-agents with isolated context
3. **Assesses** search results and plans next steps as information is gathered
4. **Synthesizes** findings with proper citations into a comprehensive final report

### Architecture

```
User ──▶ Deep Research Agent (orchestrator)
              │
              ├──▶ research-agent (subagent 1) ──▶ web search tool
              ├──▶ research-agent (subagent 2) ──▶ web search tool
              └──▶ research-agent (subagent 3) ──▶ web search tool
              │
              └──▶ Final synthesized report with citations
```

### Install Deep Agents

In [7]:
%pip install deepagents langchain-mcp-adapters markdownify

Note: you may need to restart the kernel to use updated packages.


### Set up the Web Search Tool as MCP Tool

In [8]:
from langchain_mcp_adapters.client import MultiServerMCPClient
from langchain.agents import create_agent

# Connect to the remote MCP server over Streamable HTTP
mcp_client_web_search = MultiServerMCPClient(
    {
        "web-search": {
            "url": f"http://{aca_mcp_server_open_web_search_fqdn}/mcp",
            "transport": "http",
        }
    }
)

# Discover tools exposed by the MCP server
mcp_tools_web_search = await mcp_client_web_search.get_tools()
print("MCP tools discovered:", [t.name for t in mcp_tools_web_search])

# Create an agent with the self-hosted LLM + MCP tools
agent_with_mcp = create_agent(model, mcp_tools_web_search)

MCP tools discovered: ['search', 'fetchLinuxDoArticle', 'fetchCsdnArticle', 'fetchGithubReadme', 'fetchWebContent', 'fetchJuejinArticle']


### Streaming with MCP Tools

Stream the agent's step-by-step reasoning as it decides to call the remote MCP web search tool and synthesizes the results.

In [16]:
async for step in agent_with_mcp.astream(
    {"messages": [HumanMessage(content="""
                               What is the current price for Microsoft stock ? 
                               Search the web if you don't know and fetch and analyse the data from multiple sources.
                               If a fetch operation fails, skip it and move to the next source.
                               Limit your search to only 3 sources.
    """)]},
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

================================ Human Message =================================


                               What is the current price for Microsoft stock ? 
                               Search the web if you don't know and fetch and analyse the data from multiple sources.
                               If a fetch operation fails, skip it and move to the next source.
                               Limit your search to only 3 sources.
    
================================== Ai Message ==================================
Tool Calls:
  search (call_rt2G4W3ilgcEW7ecmBFz8H6Y)
 Call ID: call_rt2G4W3ilgcEW7ecmBFz8H6Y
  Args:
    query: Microsoft stock price MSFT current price
    limit: 3
    searchMode: auto
    engines: ['duckduckgo']
================================= Tool Message =================================
Name: search

[{'type': 'text', 'text': '{\n  "query": "Microsoft stock price MSFT current price",\n  "engines": [\n    "duckduckgo"\n  ],\n  "totalResults": 3,\n  "results"

ToolException: Failed to fetch web content: Request failed with status code 500

### Define the Web Search Tool

The `tavily_search` tool uses Tavily for URL discovery, then fetches full webpage content so the agent can analyze complete sources instead of summaries.

In [13]:
import os
from typing import Annotated, Literal

import httpx
from langchain.tools import InjectedToolArg, tool
from markdownify import markdownify

def fetch_webpage_content(url: str, timeout: float = 10.0) -> str:
    """Fetch webpage and convert HTML to markdown."""
    headers = {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36"
    }
    try:
        response = httpx.get(url, headers=headers, timeout=timeout)
        response.raise_for_status()
        return markdownify(response.text)
    except Exception as e:
        return f"Error fetching {url}: {e!s}"


# @tool(parse_docstring=True)
# def web_search(
#     query: str,
#     max_results: Annotated[int, InjectedToolArg] = 1,
#     topic: Annotated[
#         Literal["general", "news", "finance"], InjectedToolArg
#     ] = "general",
# ) -> str:
#     """Search the web for information on a given query.

#     Uses Tavily to discover relevant URLs, then fetches and returns full webpage content as markdown.

#     Args:
#         query: Search query to execute
#         max_results: Maximum number of results to return (default: 1)
#         topic: Topic filter - 'general', 'news', or 'finance' (default: 'general')

#     Returns:
#         Formatted search results with full webpage content
#     """
#     search_results = tavily_client.search(
#         query,
#         max_results=max_results,
#         topic=topic,
#     )
#     result_texts = []
#     for result in search_results.get("results", []):
#         url = result["url"]
#         title = result["title"]
#         content = fetch_webpage_content(url)
#         result_texts.append(f"## {title}\n**URL:** {url}\n\n{content}\n---")

#     return f"Found {len(result_texts)} result(s) for '{query}':\n\n" + "\n".join(
#         result_texts
#     )

# print("tavily_search tool defined.")

### Define Prompt Templates

The deep research agent uses three prompt templates:

1. **Research Workflow Instructions** — Guides the orchestrator on how to plan, delegate, synthesize, and write reports.
2. **Researcher Instructions** — Tells sub-agents how to conduct web searches and format findings with citations.
3. **Sub-Agent Delegation Instructions** — Controls how the orchestrator decomposes tasks and parallelizes sub-agent work.

In [14]:
RESEARCH_WORKFLOW_INSTRUCTIONS = """# Research Workflow

Follow this workflow for all research requests:

1. **Plan**: Create a todo list with write_todos to break down the research into focused tasks
2. **Save the request**: Use write_file() to save the user's research question to `/research_request.md`
3. **Research**: Delegate research tasks to sub-agents using the task() tool - ALWAYS use sub-agents for research, never conduct research yourself
4. **Synthesize**: Review all sub-agent findings and consolidate citations (each unique URL gets one number across all findings)
5. **Write Report**: Write a comprehensive final report to `/final_report.md` (see Report Writing Guidelines below)
6. **Verify**: Read `/research_request.md` and confirm you've addressed all aspects with proper citations and structure

## Research Planning Guidelines
- Batch similar research tasks into a single TODO to minimize overhead
- For simple fact-finding questions, use 1 sub-agent
- For comparisons or multi-faceted topics, delegate to multiple parallel sub-agents
- Each sub-agent should research one specific aspect and return findings

## Report Writing Guidelines

When writing the final report to `/final_report.md`, follow these structure patterns:

**For comparisons:**
1. Introduction
2. Overview of topic A
3. Overview of topic B
4. Detailed comparison
5. Conclusion

**For lists/rankings:**
Simply list items with details - no introduction needed:
1. Item 1 with explanation
2. Item 2 with explanation
3. Item 3 with explanation

**For summaries/overviews:**
1. Overview of topic
2. Key concept 1
3. Key concept 2
4. Key concept 3
5. Conclusion

**General guidelines:**
- Use clear section headings (## for sections, ### for subsections)
- Write in paragraph form by default - be text-heavy, not just bullet points
- Do NOT use self-referential language ("I found...", "I researched...")
- Write as a professional report without meta-commentary
- Each section should be comprehensive and detailed
- Use bullet points only when listing is more appropriate than prose

**Citation format:**
- Cite sources inline using [1], [2], [3] format
- Assign each unique URL a single citation number across ALL sub-agent findings
- End report with ### Sources section listing each numbered source
- Number sources sequentially without gaps (1,2,3,4...)
- Format: [1] Source Title: URL (each on separate line for proper list rendering)
- Example:

 Some important finding [1]. Another key insight [2].

 ### Sources
 [1] AI Research Paper: https://example.com/paper
 [2] Industry Analysis: https://example.com/analysis
"""

print("RESEARCH_WORKFLOW_INSTRUCTIONS defined.")

RESEARCH_WORKFLOW_INSTRUCTIONS defined.


In [15]:
RESEARCHER_INSTRUCTIONS = """You are a research assistant conducting research on the user's input topic. For context, today's date is {date}.

Your job is to use tools to gather information about the user's input topic.
You can use the tavily_search tool to find resources that can help answer the research question.
You can call it in series or in parallel, your research is conducted in a tool-calling loop.

You have access to the tavily_search tool for conducting web searches.

Think like a human researcher with limited time. Follow these steps:

1. **Read the question carefully** - What specific information does the user need?
2. **Start with broader searches** - Use broad, comprehensive queries first
3. **After each search, pause and assess** - Do I have enough to answer? What's still missing?
4. **Execute narrower searches as you gather information** - Fill in the gaps
5. **Stop when you can answer confidently** - Don't keep searching for perfection

**Tool Call Budgets** (Prevent excessive searching):
- **Simple queries**: Use 2-3 search tool calls maximum
- **Complex queries**: Use up to 5 search tool calls maximum
- **Always stop**: After 5 search tool calls if you cannot find the right sources

**Stop Immediately When**:
- You can answer the user's question comprehensively
- You have 3+ relevant examples/sources for the question
- Your last 2 searches returned similar information

After each search, assess results before continuing: What key information did I find? What's missing? Do I have enough to answer? Should I search more or provide my answer?

When providing your findings back to the orchestrator:

1. **Structure your response**: Organize findings with clear headings and detailed explanations
2. **Cite sources inline**: Use [1], [2], [3] format when referencing information from your searches
3. **Include Sources section**: End with ### Sources listing each numbered source with title and URL

Example:
## Key Findings

Context engineering is a critical technique for AI agents [1]. Studies show that proper context management can improve performance by 40% [2].

### Sources
[1] Context Engineering Guide: https://example.com/context-guide
[2] AI Performance Study: https://example.com/study

The orchestrator will consolidate citations from all sub-agents into the final report.
"""

print("RESEARCHER_INSTRUCTIONS defined.")

RESEARCHER_INSTRUCTIONS defined.


In [19]:
SUBAGENT_DELEGATION_INSTRUCTIONS = """# Sub-Agent Research Coordination

Your role is to coordinate research by delegating tasks from your TODO list to specialized research sub-agents.

## Delegation Strategy

**DEFAULT: Start with 1 sub-agent** for most queries:
- "What is quantum computing?" -> 1 sub-agent (general overview)
- "List the top 10 coffee shops in San Francisco" -> 1 sub-agent
- "Summarize the history of the internet" -> 1 sub-agent
- "Research context engineering for AI agents" -> 1 sub-agent (covers all aspects)

**ONLY parallelize when the query EXPLICITLY requires comparison or has clearly independent aspects:**

**Explicit comparisons** -> 1 sub-agent per element:
- "Compare OpenAI vs Anthropic vs DeepMind AI safety approaches" -> 3 parallel sub-agents
- "Compare Python vs JavaScript for web development" -> 2 parallel sub-agents

**Clearly separated aspects** -> 1 sub-agent per aspect (use sparingly):
- "Research renewable energy adoption in Europe, Asia, and North America" -> 3 parallel sub-agents (geographic separation)
- Only use this pattern when aspects cannot be covered efficiently by a single comprehensive search

## Key Principles
- **Bias towards single sub-agent**: One comprehensive research task is more token-efficient than multiple narrow ones
- **Avoid premature decomposition**: Don't break "research X" into "research X overview", "research X techniques", "research X applications" - just use 1 sub-agent for all of X
- **Parallelize only for clear comparisons**: Use multiple sub-agents when comparing distinct entities or geographically separated data

## Parallel Execution Limits
- Use at most {max_concurrent_research_units} parallel sub-agents per iteration
- Make multiple task() calls in a single response to enable parallel execution
- Each sub-agent returns findings independently

## Research Limits
- Stop after {max_researcher_iterations} delegation rounds if you haven't found adequate sources
- Stop when you have sufficient information to answer comprehensively
- Bias towards focused research over exhaustive exploration"""

### Create the Deep Research Agent

The `create_deep_agent` function from the `deepagents` package creates the orchestrator agent with:

- The self-hosted LLM model
- The `tavily_search` tool (available to both orchestrator and sub-agents)
- A combined system prompt (workflow + delegation instructions)
- A research sub-agent definition with its own prompt and tools

In [ ]:
from datetime import datetime
from deepagents import create_deep_agent
from langchain.chat_models import init_chat_model
from langchain_azure_ai.chat_models import AzureAIOpenAIApiChatModel

max_concurrent_research_units = 3
max_researcher_iterations = 3

current_date = datetime.now().strftime("%Y-%m-%d")

# Combine workflow + delegation instructions for the orchestrator
INSTRUCTIONS = (
    RESEARCH_WORKFLOW_INSTRUCTIONS
    + "\n\n"
    + "=" * 80
    + "\n\n"
    + SUBAGENT_DELEGATION_INSTRUCTIONS.format(
        max_concurrent_research_units=max_concurrent_research_units,
        max_researcher_iterations=max_researcher_iterations,
    )
)

# Define the research sub-agent
research_sub_agent = {
    "name": "research-agent",
    "description": "Delegate research to the sub-agent. Give one topic at a time.",
    "system_prompt": RESEARCHER_INSTRUCTIONS.format(date=current_date),
    "tools": [mcp_client_web_search],
}

init_chat_model = init_chat_model(model="azure_ai:gpt-5.4", temperature=0.0)

# Create the deep research agent using the self-hosted LLM
agent = create_deep_agent(
    model=init_chat_model,
    tools=[mcp_tools_web_search],
    system_prompt=INSTRUCTIONS,
    subagents=[research_sub_agent],
)

print("Deep Research Agent created successfully!")

ImportError: Initializing AzureAIOpenAIApiChatModel requires the langchain-azure-ai package. Please install it with `pip install langchain-azure-ai`

### Run the Deep Research Agent — Synchronous

Ask the agent a research question. It will plan, delegate to sub-agents, and synthesize a report.

In [ ]:
from langchain_core.messages import HumanMessage

result = await agent.ainvoke(
    {
        "messages": [
            HumanMessage(
                content="What are the main differences between RAG and fine-tuning for LLM applications?"
            )
        ]
    }
)

# Print the final response
for msg in result.get("messages", []):
    if hasattr(msg, "content") and msg.content:
        msg.pretty_print()

### Run with Streaming

Stream updates as the agent works — each step (planning, sub-agent delegation, tool calls, synthesis) is printed in real time.

In [ ]:
async for step in agent.astream(
    {
        "messages": [
            HumanMessage(
                content="Compare Azure Container Apps vs Azure Kubernetes Service for deploying AI workloads."
            )
        ]
    },
    stream_mode="values",
):
    step["messages"][-1].pretty_print()

## More Resources

- [Build a Deep Research Agent](https://docs.langchain.com/oss/python/deepagents/deep-research) — official LangChain tutorial
- [Deep Agents](https://docs.langchain.com/oss/python/deepagents) — orchestrate subagents with context isolation
- [Subagents](https://docs.langchain.com/oss/python/deepagents/subagents) — configure subagents with different tools and prompts
- [Deep Research Course](https://academy.langchain.com/courses/deep-research-with-langgraph) — full course on deep research with LangGraph
- [Tavily](https://www.tavily.com/) — web search API for AI agents
- [Full Deep Research Example on GitHub](https://github.com/langchain-ai/deepagents/tree/main/examples/deep_research)